**Суркова Татьяна**

## Домашнее задание 4

Задание 1 (синтаксис)

Напишите функцию check_car_id(car_id: str) → str.
 
 Формат номера: 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона.
 Буквы только из набора, допустимого в РФ-номерах: А, В, Е, К, М, Н, О, Р, С, Т, У, Х (кириллица). Учитывайте путаницу с латиницей на входе - нормализуйте или явно опишите правило.
 
 • Если номер валиден: «Номер А222ВС валиден. Регион: 96.»
 • Если невалиден: «Номер не валиден.»

Пример вывода:

check_car_id('А222ВС96')
 Номер А222ВС валиден. Регион: 96.
 
check_car_id('А123ВЕ777')
 Номер А123ВЕ валиден. Регион: 777.
 
check_car_id('АБ22ВВ193')
 Номер не валиден.

Пограничные случаи:

check_car_id('A222BC96')   # латиница A/B/C - по правилу нормализации
 Номер А222ВС валиден. Регион: 96.
 либо «Номер не валиден.» - если нормализацию не делаете; зафиксируйте выбор
 
check_car_id('А222ВС9')    # регион из 1 цифры
 Номер не валиден.
 
check_car_id('А222ВС9699') # регион из 4 цифр
 Номер не валиден.
 
check_car_id('Я222ВС96')   # недопустимая буква Я
 Номер не валиден.
 
check_car_id('')
 Номер не валиден.

In [ ]:
import re


def check_car_id(car_id: str) -> str:
    car_id = car_id.translate(str.maketrans({
        'A': 'А',
        'B': 'В',
        'C': 'С',
    }))

    # 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона
    pattern = r'^([АВЕКМНОРСТУХ])(\d{3})([АВЕКМНОРСТУХ]{2})(\d{2,3})$'

    match = re.fullmatch(pattern, car_id)

    if not match:
        return 'Номер не валиден.'

    letter, digits, letters, region = match.groups()
    number = letter + digits + letters

    return f'Номер {number} валиден. Регион: {region}.'


print(check_car_id('А222ВС96'))
print(check_car_id('А123ВЕ777'))
print(check_car_id('АБ22ВВ193'))
print(check_car_id('A222BC96'))
print(check_car_id('А222ВС9'))
print(check_car_id('А222ВС9699'))
print(check_car_id('Я222ВС96'))
print(check_car_id(''))

Номер А222ВС валиден. Регион: 96.
Номер А123ВЕ валиден. Регион: 777.
Номер не валиден.
Номер А222ВС валиден. Регион: 96.
Номер не валиден.
Номер не валиден.
Номер не валиден.
Номер не валиден.


Задание 2 (повышенная сложность)

Дан многострочный лог. Реализуйте parse_logs(text: str) → list[dict].
 
 1. Одной (или составной) регуляркой извлеките строки только в формате:
    YYYY-MM-DD HH:MM:SS LEVEL ...
 2. Из хвоста строки достаньте user, action, ip; если есть card=... - тоже.
 3. Валидация через regex:
    • email (разумный практический паттерн);
    • IPv4;
    • карта: маска XXXX-XXXX-XXXX-XXXX; в результате хранить только ****-****-****-1111 (последние 4 цифры).
 4. Невалидные поля не роняют парсер: в словаре "errors": ["invalid_email", ...].
 5. Бонус: удалите последовательные повторы слов в текстовом поле сообщения через re.

Входной пример:

2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12

2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12

2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111

Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5

Пограничные случаи:

Пустая строка / только пробелы → [].

Строка без user=/action=/ip= → запись с отсутствующими полями (None) и соответствующими errors, либо пропуск - зафиксируйте правило.

ip=256.0.0.1 или ip=1.2.3 → errors содержит invalid_ip.

card=4111111111111111 без дефисов → не распознаётся как карта (card=None) или invalid_card — зафиксируйте правило.

Повторы слов в сообщении: «error error error timeout» → «error timeout» (бонус).

In [3]:
def parse_logs(text: str) -> list[dict]:
    result = []

    # Регулярка для строк лога
    log_pattern = re.compile(
        r'^'
        r'(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})'
        r'\s+'
        r'(?P<level>INFO|WARN|ERROR|DEBUG)'
        r'\s+'
        r'(?P<message>.*)'
        r'$'
    )

    # Email
    email_pattern = re.compile(
        r'^[A-Za-z0-9.!#$%&\'*+/=?^_`{|}~-]+'
        r'@'
        r'[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+$'
    )

    # IPv4
    ip_pattern = re.compile(
        r'^(?:'
        r'(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.'
        r'){3}'
        r'(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)'
        r'$'
    )

    # Карта строго в формате XXXX-XXXX-XXXX-XXXX
    card_pattern = re.compile(
        r'^\d{4}-\d{4}-\d{4}-\d{4}$'
    )

    # Ищем пары key=value
    field_pattern = re.compile(
        r'(\w+)=([^\s]+)'
    )

    # Для удаления последовательных повторов слов
    repeat_pattern = re.compile(
        r'\b(\w+)(?:\s+\1\b)+',
        re.IGNORECASE
    )

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        match = log_pattern.fullmatch(line)

        # Строки с неправильным форматом даты/времени пропускаем
        if not match:
            continue

        ts = match.group('ts')
        level = match.group('level')
        message = match.group('message')

        # Извлекаем key=value
        fields = dict(field_pattern.findall(message))

        user = fields.get('user')
        action = fields.get('action')
        ip = fields.get('ip')
        card = fields.get('card')

        errors = []

        # Проверяем наличие обязательных полей
        if user is None:
            errors.append('missing_user')
        elif not email_pattern.fullmatch(user):
            errors.append('invalid_email')

        if action is None:
            errors.append('missing_action')

        if ip is None:
            errors.append('missing_ip')
        elif not ip_pattern.fullmatch(ip):
            errors.append('invalid_ip')

        # Проверяем карту
        if card is not None:
            if card_pattern.fullmatch(card):
                card = '****-****-****-' + card[-4:]
            else:
                # По выбранному правилу неправильная карта
                # считается ошибкой
                errors.append('invalid_card')
                card = None

        # Удаляем последовательные повторы слов в message
        cleaned_message = repeat_pattern.sub(r'\1', message)

        result.append({
            'ts': ts,
            'level': level,
            'user': user,
            'action': action,
            'ip': ip,
            'card': card,
            'message': cleaned_message,
            'errors': errors
        })

    return result

with open('logs_hw4.txt', 'r', encoding='utf-8') as file:
    text = file.read()

result = parse_logs(text)

print(result)

[{'ts': '2024-03-15 10:51:25', 'level': 'WARN', 'user': 'user+tag@gmail.com', 'action': 'login', 'ip': '203.0.113.42', 'card': None, 'message': 'user=user+tag@gmail.com action=login ip=203.0.113.42 extra=foo note=bar', 'errors': []}, {'ts': '2024-13-01 12:00:00', 'level': 'INFO', 'user': 'ivan@mail.ru', 'action': 'login', 'ip': '1.1.1.1', 'card': None, 'message': 'user=ivan@mail.ru action=login ip=1.1.1.1', 'errors': []}, {'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'login', 'ip': '8.8.8.8', 'card': None, 'message': 'user=petr.sidorov@yandex.ru action=login ip=8.8.8.8 msg=retry please', 'errors': []}, {'ts': '2024-03-15 09:36:54', 'level': 'WARN', 'user': 'petr.sidorov@yandex.ru', 'action': 'retry', 'ip': '999.1.1.1', 'card': None, 'message': 'user=petr.sidorov@yandex.ru action=retry ip=999.1.1.1', 'errors': ['invalid_ip']}, {'ts': '2024-03-15 09:07:09', 'level': 'ERROR', 'user': 'user@domain.', 'action': 'delete', 'ip': '0.0.0.0', 'card':